# SDT DPO pilot V2 — multi-sample fresh-response evaluation

This notebook never trains. It evaluates the three saved matched generation
samples from the V2 run with three stronger local judges. Every pair is judged
blindly in both A/B orders. Failed and order-inconsistent votes cannot create a
win. Final aggregation is first across judges for each generation seed, then
across generation seeds for each unique prompt.


## 1. GPU, exact branch, and installation


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import subprocess, sys, json, hashlib, re
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU."
subprocess.run(["nvidia-smi"], check=True)
REPO_URL = "https://github.com/Rana-Ezzeddine/SDT.git"
BRANCH = "1500-record-dpo-pipeline"
REPO_DIR = Path("/content/SDT")
if REPO_DIR.exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "switch", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", GIT_COMMIT)


## 2. Select and verify the completed V2 run


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/SDT_DPO_Pilot_V2")
RUN_NAME = ""  # Paste the exact RUN_NAME printed by the training notebook; blank selects latest complete run.
if RUN_NAME:
    DRIVE_RUN = DRIVE_ROOT / RUN_NAME
else:
    candidates = sorted(
        [p for p in DRIVE_ROOT.glob("pilot-1500-v2-*") if (p / "artifact-manifest.json").exists()],
        key=lambda p: p.stat().st_mtime,
    )
    assert candidates, "No completed V2 run found."
    DRIVE_RUN = candidates[-1]
    RUN_NAME = DRIVE_RUN.name
assert (DRIVE_RUN / "artifact-manifest.json").exists(), "Training/generation final audit is missing."
MODEL_DIR = DRIVE_RUN / "training" / "model"
INTEGRITY = json.loads((DRIVE_RUN / "model-integrity.json").read_text())
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

for item in INTEGRITY["weights"]:
    path = MODEL_DIR / item["name"]
    assert path.exists() and path.stat().st_size == item["bytes"], f"Missing/corrupt weight: {path}"
    assert sha256_file(path) == item["sha256"], f"Weight hash mismatch: {path}"
print("Verified V2 run:", DRIVE_RUN)
print("Saved model bytes:", INTEGRITY["total_weight_bytes"])


## 3. Verify all matched generation samples


In [ ]:
GENERATION_SEEDS = [42, 202, 303]
GENERATION_ROOT = DRIVE_RUN / "test" / "fresh-generations"
generation_files = {}
for seed in GENERATION_SEEDS:
    baseline = GENERATION_ROOT / f"seed-{seed}" / "baseline-generations.jsonl"
    dpo = GENERATION_ROOT / f"seed-{seed}" / "dpo-generations.jsonl"
    def read_rows(path):
        rows = [json.loads(x) for x in path.read_text().splitlines() if x.strip()]
        assert rows and len({str(x["prompt_id"]) for x in rows}) == len(rows)
        assert all(str(x.get("response", "")).strip() for x in rows)
        return rows
    baseline_rows, dpo_rows = read_rows(baseline), read_rows(dpo)
    assert [(str(x["prompt_id"]), x["prompt"]) for x in baseline_rows] == [(str(x["prompt_id"]), x["prompt"]) for x in dpo_rows]
    generation_files[seed] = {"baseline": baseline, "dpo": dpo}
    identical = sum(" ".join(a["response"].split()) == " ".join(b["response"].split()) for a, b in zip(baseline_rows, dpo_rows))
    print({"seed": seed, "prompts": len(baseline_rows), "identical": identical})


## 4. Blind stronger-judge panel with position reversal

The models run sequentially, so only one judge occupies GPU memory at a time.
Outputs are written directly to Drive and resume row-by-row. On an A100 40/80 GB,
the default 12–14B judges should fit in bfloat16. If a model is unavailable, use a
comparably capable ungated instruct model and start a new evaluation version.


In [ ]:
JUDGE_MODELS = [
    "Qwen/Qwen2.5-14B-Instruct",
    "mistralai/Mistral-Nemo-Instruct-2407",
    "allenai/OLMo-2-1124-13B-Instruct",
]
EVAL_ROOT = DRIVE_RUN / "test" / "fresh-evaluation-v2"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

def judge_slug(index, model):
    short = re.sub(r"[^a-z0-9]+", "-", model.lower()).strip("-")[-56:]
    return f"judge-{index + 1}-{short}"

COMBINED_ROOT = EVAL_ROOT / "combined-inputs"
COMBINED_ROOT.mkdir(parents=True, exist_ok=True)
combined_files = {}
for model_name in ("baseline", "dpo"):
    combined = COMBINED_ROOT / f"{model_name}-generations.jsonl"
    with combined.open("w", encoding="utf-8") as handle:
        for seed in GENERATION_SEEDS:
            for line in generation_files[seed][model_name].read_text().splitlines():
                if not line.strip():
                    continue
                row = json.loads(line)
                original_prompt_id = str(row["prompt_id"])
                row["original_prompt_id"] = original_prompt_id
                row["generation_seed"] = seed
                row["prompt_id"] = f"{seed}::{original_prompt_id}"
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    combined_files[model_name] = combined

# Combining the three generation seeds lets each 12–14B judge load only twice
# (forward and reverse), rather than six times. Split files below restore the
# original prompt IDs for seed-aware aggregation.
manifest_entries = []
for judge_index, judge_model in enumerate(JUDGE_MODELS):
    judge_root = EVAL_ROOT / "judgments" / judge_slug(judge_index, judge_model)
    judge_root.mkdir(parents=True, exist_ok=True)
    combined_details = {}
    for orientation, reverse in [("forward", False), ("reverse", True)]:
        details = judge_root / f"combined-{orientation}.jsonl"
        command = [
            "sdt-judge-generations-local",
            "--baseline", str(combined_files["baseline"]),
            "--dpo", str(combined_files["dpo"]),
            "--details", str(details),
            "--summary", str(judge_root / f"combined-{orientation}-summary.json"),
            "--failures", str(judge_root / f"combined-{orientation}-failures.jsonl"),
            "--judge-model", judge_model, "--seed", "42",
            "--max-new-tokens", "512", "--max-retries", "3", "--continue-on-failure",
        ]
        if reverse:
            command.append("--reverse-order")
        print("Running", judge_model, orientation, "across all generation seeds")
        subprocess.run(command, check=True)
        combined_details[orientation] = details

    for seed in GENERATION_SEEDS:
        seed_dir = judge_root / f"seed-{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        paths = {}
        prefix = f"{seed}::"
        for orientation in ("forward", "reverse"):
            split_path = seed_dir / f"{orientation}.jsonl"
            with split_path.open("w", encoding="utf-8") as handle:
                for line in combined_details[orientation].read_text().splitlines():
                    row = json.loads(line)
                    combined_id = str(row["prompt_id"])
                    if combined_id.startswith(prefix):
                        row["prompt_id"] = combined_id[len(prefix):]
                        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
            assert split_path.exists() and split_path.stat().st_size > 0
            paths[orientation] = str(split_path)
        manifest_entries.append({"generation_seed": seed, "judge_model": judge_model, **paths})

MANIFEST = EVAL_ROOT / "evaluation-manifest.json"
MANIFEST.write_text(json.dumps({
    "run_name": RUN_NAME, "generation_seeds": GENERATION_SEEDS,
    "judge_models": JUDGE_MODELS, "position_reversal": True,
    "failed_votes_treated_as_unavailable": True, "judgments": manifest_entries,
}, indent=2) + "\n")
print("Saved:", MANIFEST)


## 5. Aggregate at the unique-prompt level


In [ ]:
FINAL_SUMMARY = EVAL_ROOT / "fresh-evaluation-summary.json"
PROMPT_DETAILS = EVAL_ROOT / "fresh-evaluation-prompt-details.jsonl"
subprocess.run([
    "sdt-aggregate-fresh-evaluation", "--manifest", str(MANIFEST),
    "--output", str(FINAL_SUMMARY), "--details", str(PROMPT_DETAILS),
    "--bootstrap-samples", "10000",
], check=True)
report = json.loads(FINAL_SUMMARY.read_text())
print(json.dumps({k: report[k] for k in [
    "n_prompts", "prompt_level_dpo_wins", "prompt_level_baseline_wins",
    "prompt_level_ties", "prompt_macro_tie_adjusted_dpo_score", "prompt_bootstrap_95",
]}, indent=2))


## 6. Results and diagnostics


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

display(pd.DataFrame([{"judge": judge, **metrics} for judge, metrics in report["judge_diagnostics"].items()]))
display(pd.DataFrame([{
    "prompts": report["n_prompts"], "DPO wins": report["prompt_level_dpo_wins"],
    "Baseline wins": report["prompt_level_baseline_wins"], "Ties": report["prompt_level_ties"],
    "Tie-adjusted DPO score": report["prompt_macro_tie_adjusted_dpo_score"],
    "95% low": report["prompt_bootstrap_95"][0], "95% high": report["prompt_bootstrap_95"][1],
}]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["DPO", "Baseline", "Tie"], [report["prompt_level_dpo_wins"], report["prompt_level_baseline_wins"], report["prompt_level_ties"]], color=["#2E86AB", "#D1495B", "#999999"])
axes[0].set_title("Prompt-level fresh-response outcomes")
names = list(report["dimension_deltas"])
values = [report["dimension_deltas"][name]["mean_dpo_minus_baseline"] for name in names]
axes[1].barh(names, values, color=["#2E86AB" if value >= 0 else "#D1495B" for value in values])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Mean judge-score delta: DPO − baseline")
plt.tight_layout()
FIGURE = EVAL_ROOT / "fresh-evaluation-overview.png"
plt.savefig(FIGURE, dpi=180, bbox_inches="tight")
plt.show()


## 7. Final evaluation audit


In [ ]:
required = [MANIFEST, FINAL_SUMMARY, PROMPT_DETAILS, FIGURE]
required += [Path(entry[key]) for entry in manifest_entries for key in ("forward", "reverse")]
missing = [str(path) for path in required if not path.exists() or path.stat().st_size == 0]
assert not missing, "Missing evaluation artifacts:\n" + "\n".join(missing)
AUDIT = EVAL_ROOT / "evaluation-artifact-manifest.json"
AUDIT.write_text(json.dumps({
    "run_name": RUN_NAME, "git_commit": GIT_COMMIT,
    "completed_at_utc": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
    "artifacts": [{"path": str(p.relative_to(DRIVE_RUN)), "bytes": p.stat().st_size} for p in required],
}, indent=2) + "\n")
os.sync()
print("V2 evaluation complete:", EVAL_ROOT)
print("Final result:", FINAL_SUMMARY)
